# Redrob Hackathon — Candidate Ranking Pipeline

**Goal:** Rank top 100 from 100K candidates for Senior AI Engineer.

**Runtime:** GPU (T4+) — *Runtime → Change runtime type → T4 GPU*

### Pipeline
1. Clone repo + extract dataset
2. Install deps
3. Pre-compute: bi-encoder embeddings + BM25 + cross-encoder re-ranking + features
4. Rank: XGBoost LTR → top 100 with reasoning
5. Validate + download

## Step 1 — GPU Check + Clone + Extract

In [3]:
# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → T4 GPU")
print(result.stdout.split('\n')[0])
print("GPU OK")

Wed Jul  1 00:14:09 2026       
GPU OK


In [4]:
# Clone repo
!git clone https://github.com/JASHWANTHS07/india-runs.git
%cd india-runs

Cloning into 'india-runs'...
remote: Enumerating objects: 267, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 267 (delta 85), reused 102 (delta 42), pack-reused 102 (from 1)
Receiving objects: 100% (267/267), 75.79 MiB | 22.73 MiB/s, done.
Resolving deltas: 100% (124/124), done.
/content/india-runs


In [5]:
# Extract dataset from Data.rar
!apt-get install -qq unrar
!unrar x -o+ dataset/Data.rar dataset/
print("\nExtraction complete.")
!ls dataset/India_runs_data_and_ai_challenge/


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from dataset/Data.rar

Extracting  dataset/.DS_Store                                              0%  OK 
Creating    dataset/India_runs_data_and_ai_challenge                  OK
Extracting  dataset/India_runs_data_and_ai_challenge/candidates.jsonl      22% 44% 66% 89% 99%  OK 
Extracting  dataset/India_runs_data_and_ai_challenge/candidate_schema.json      99%  OK 
Extracting  dataset/India_runs_data_and_ai_challenge/job_description.docx      99%  OK 
Extracting  dataset/India_runs_data_and_ai_challenge/sample_candidates.json      99%  OK 
Extracting  dataset/India_runs_data_and_ai_challenge/sample_submission.csv      99%  OK 
All OK

Extraction complete.
candidate_schema.json  job_description.docx    sample_submission.csv
candidates.jsonl       sample_candidates.json


## Step 2 — Install Dependencies

In [6]:
# Install deps (torch is pre-installed in Colab — don't reinstall)
!pip install -q sentence-transformers tqdm pandas pyarrow xgboost
print("Dependencies installed.")

Dependencies installed.


In [7]:
# Verify dataset
import os
CANDIDATES = "dataset/India_runs_data_and_ai_challenge/candidates.jsonl"
size_mb = os.path.getsize(CANDIDATES) / 1e6
print(f"candidates.jsonl: {size_mb:.1f} MB")
assert size_mb > 400, f"Too small ({size_mb:.1f} MB)"
print("Dataset OK")

candidates.jsonl: 487.3 MB
Dataset OK


## Step 3 — Pre-computation (GPU)

Runs three retrieval stages:
1. **Bi-encoder** — BAAI/bge-large-en-v1.5 embeddings for all 100K candidates
2. **BM25** — sparse keyword matching against JD
3. **Cross-encoder** — ms-marco-MiniLM-L-6-v2 re-ranks top-1000 by bi-encoder score

Plus 78 structured features (v2: headline, salary fit, assessment depth, market demand, template detection, career coherence).

**Expected time:** ~20–30 min on T4.

In [ ]:
!python src/precompute.py \
    --candidates dataset/India_runs_data_and_ai_challenge/candidates.jsonl \
    --artifacts  ./artifacts

Loading candidates from dataset/India_runs_data_and_ai_challenge/candidates.jsonl...
Extracting features: 100000it [01:09, 1449.10it/s]
  100000 candidates loaded, features extracted

Computing BM25 scores...
  BM25 done in 19.9s

Loading bi-encoder: BAAI/bge-large-en-v1.5 (device: cuda)
modules.json: 100% 349/349 [00:00<00:00, 1.88MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 900kB/s]
README.md: 100% 94.6k/94.6k [00:00<00:00, 120MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 200kB/s]
config.json: 100% 779/779 [00:00<00:00, 5.25MB/s]
model.safetensors: 100% 1.34G/1.34G [00:08<00:00, 161MB/s]
Loading weights: 100% 391/391 [00:00<00:00, 5998.68it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 1.20MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 19.8MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 76.3MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 495kB/s]
config.json: 100% 191/191 [00:00<00:00, 1.29MB/s]
Embedding JD query...
Embedding

In [7]:
# Verify artifacts
import numpy as np
import pandas as pd

emb = np.load('artifacts/embeddings.npy')
jd  = np.load('artifacts/jd_embedding.npy')
df  = pd.read_parquet('artifacts/features.parquet')

assert emb.shape[1] == 1024, f"Unexpected dim: {emb.shape}"
assert 'bm25_score' in df.columns, "Missing bm25_score"
assert 'cross_encoder_score' in df.columns, "Missing cross_encoder_score"

print(f"embeddings.npy       : {emb.shape}  ({emb.nbytes/1e6:.0f} MB)")
print(f"jd_embedding         : {jd.shape}")
print(f"features.parquet     : {len(df)} rows, {len(df.columns)} cols")
print(f"  bm25 range         : [{df.bm25_score.min():.3f}, {df.bm25_score.max():.3f}]")
print(f"  cross-encoder range: [{df.cross_encoder_score.min():.3f}, {df.cross_encoder_score.max():.3f}]")
print("Artifacts OK")

embeddings.npy       : (100000, 1024)  (410 MB)
jd_embedding         : (1024,)
features.parquet     : 100000 rows, 87 cols
  bm25 range         : [0.001, 1.000]
  cross-encoder range: [0.000, 1.000]
Artifacts OK


In [ ]:
from google.colab import files
files.download('artifacts/embeddings.npy')
files.download('artifacts/jd_embedding.npy')
files.download('artifacts/features.parquet')
print("Downloaded.")

## Step 4 — Rank (CPU, XGBoost LTR)

Multiplicative scoring hierarchy: retrieval depth as primary discriminator.
Trains XGBoost (rank:pairwise) on pseudo-relevance labels using 78 features
plus semantic, BM25, and cross-encoder scores. Sigmoid normalization for wider score spread.

**Expected time:** <3 min.

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 16.5 MB/s eta 0:00:00


In [2]:
!python src/rank.py \
    --artifacts ./artifacts \
    --out       ./jashwanth_s_new78.csv \
    --method    xgboost \
    --tune      --tune-trials 70

python3: can't open file '/content/src/rank.py': [Errno 2] No such file or directory


## Step 5 — Validate & Download

In [9]:
import pandas as pd, sys
from dataclasses import fields

df = pd.read_csv('jashwanth_s.csv')
assert len(df) == 100
assert set(df['rank']) == set(range(1, 101))
scores = df.sort_values('rank')['score'].tolist()
assert all(scores[i] >= scores[i+1] for i in range(len(scores)-1)), "Scores not non-increasing"

# Check tie-break: equal scores must have candidate_id ascending
rows = df.sort_values('rank')[['rank','score','candidate_id']].values.tolist()
for i in range(len(rows)-1):
    if rows[i][1] == rows[i+1][1]:
        assert rows[i][2] < rows[i+1][2], f"Tie-break violated at ranks {rows[i][0]},{rows[i+1][0]}"

# Honeypot check
sys.path.insert(0, '.')
from src.features import CandidateFeatures
from src.honeypot import is_honeypot
feat_df = pd.read_parquet('artifacts/features.parquet')
top_feats = feat_df[feat_df['candidate_id'].isin(set(df['candidate_id']))]
honeypots = []
for _, row in top_feats.iterrows():
    kwargs = {f.name: row[f.name] for f in fields(CandidateFeatures) if f.name in row.index}
    kwargs.setdefault('profile_text', '')
    if is_honeypot(CandidateFeatures(**kwargs)):
        honeypots.append(row['candidate_id'])

print(f"Rows      : {len(df)}")
print(f"Scores    : {scores[0]:.4f} → {scores[-1]:.4f}")
print(f"Honeypots : {len(honeypots)} (must be 0)")
assert len(honeypots) == 0
print("All checks passed.")

FileNotFoundError: [Errno 2] No such file or directory: 'jashwanth_s.csv'

In [ ]:
# Top 10
pd.set_option('display.max_colwidth', 80)
df.sort_values('rank').head(10)[['rank', 'candidate_id', 'score', 'reasoning']]

In [ ]:
# Download
from google.colab import files
files.download('jashwanth_s.csv')
print("Downloaded.")